# Master Benchmark Suite: Full 10-Model Evaluation for Drug `N02BA`

Collects holdout predictions across all candidate models for drug `N02BA` on the 2019 Test set:
* **Part A: Point Forecast Accuracy Benchmark Table (Sorted by RMSLE)**
* **Part B: Enterprise Probabilistic Demand Range Deliverable ($[P_{10}, P_{50}, P_{90}]$)**


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'N02BA'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for N02BA loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Collect All Holdout Model Predictions & Display Benchmark Ladder
m0 = pd.read_csv('m0_naive_preds.csv')['pred_Naive'].values
m1 = pd.read_csv('m1_arima_preds.csv')['pred_ARIMA'].values
m2 = pd.read_csv('m2_ets_preds.csv')['pred_ETS'].values
m3a = pd.read_csv('m3_sarima_preds.csv')['pred_SARIMA'].values
m3b = pd.read_csv('m3_sarimax_preds.csv')['pred_SARIMAX'].values
m4 = pd.read_csv('m4_prophet_preds.csv')['pred_Prophet'].values
m5 = pd.read_csv('m5_lstm_preds.csv')['pred_LSTM'].values
m6 = pd.read_csv('m6_lightgbm_preds.csv')['pred_LightGBM'].values
m7 = pd.read_csv('m7_xgb_quantile_preds.csv')['pred_XGB_Quantile'].values
m8 = pd.read_csv('m8_tft_preds.csv')['pred_TFT'].values

m_hybrid = 0.80 * m6 + 0.20 * m2

model_dict = {
    'Hybrid Ensemble (LightGBM 0.8 + ETS 0.2)': m_hybrid,
    'Model 6: LightGBM + SHAP (Optuna)': m6,
    'Model 7: XGBoost Quantile (Optuna)': m7,
    'Model 2: Holt-Winters ETS': m2,
    'Model 4: Meta Prophet': m4,
    'Model 3b: SARIMAX + Exog': m3b,
    'Model 8: TFT / Deep Attention': m8,
    'Model 1: Classical ARIMA': m1,
    'Model 3a: Pure SARIMA': m3a,
    'Model 5: PyTorch LSTM': m5,
    'Model 0: Optimised Naive (k*=365)': m0
}

records = []
for name, preds in model_dict.items():
    met = evaluate_metrics(test_series.values, preds)
    met['Model'] = name
    records.append(met)

benchmark_df = pd.DataFrame(records)[['Model', 'RMSLE', 'RMSE', 'MAE', 'WAPE (%)']].sort_values('RMSLE').reset_index(drop=True)
benchmark_df.index = benchmark_df.index + 1

print("==========================================================================")
print(f"  PART A: POINT FORECAST BENCHMARK LADDER ({TARGET_DRUG} — 2019 HOLDOUT TEST SET)")
print("==========================================================================")
display(benchmark_df)

champion = benchmark_df.iloc[0]
print("\nChampion Model for Point Forecasting:")
print(f"  * #1 Rank Model : {champion['Model']}")
print(f"  * RMSLE         : {champion['RMSLE']:.6f}")
print(f"  * RMSE          : {champion['RMSE']:.4f}")
print(f"  * MAE           : {champion['MAE']:.4f}")
print(f"  * WAPE (%)      : {champion['WAPE (%)']:.2f}%")


  PART A: POINT FORECAST BENCHMARK LADDER (N02BA — 2019 HOLDOUT TEST SET)


,Model,RMSLE,RMSE,MAE,WAPE (%)
1,Hybrid Ensemble (LightGBM 0.8 + ETS 0.2),0.511368,1.982126,1.497737,47.836336
2,Model 6: LightGBM + SHAP (Optuna),0.514046,1.972742,1.503744,48.028194
3,Model 7: XGBoost Quantile (Optuna),0.523393,1.962143,1.521287,48.588510
4,Model 1: Classical ARIMA,0.523646,2.091351,1.562264,49.897283
5,Model 5: PyTorch LSTM,0.537727,1.971864,1.522080,48.613821
6,Model 2: Holt-Winters ETS,0.537991,2.156463,1.603261,51.206683
7,Model 8: TFT / Deep Attention,0.538094,1.950018,1.518667,48.504830
8,Model 3a: Pure SARIMA,0.547562,2.194525,1.639588,52.366918
9,Model 4: Meta Prophet,0.549756,2.173745,1.631876,52.120598
10,Model 3b: SARIMAX + Exog,0.565082,2.249072,1.702438,54.374296



Champion Model for Point Forecasting:
  * #1 Rank Model : Hybrid Ensemble (LightGBM 0.8 + ETS 0.2)
  * RMSLE         : 0.511368
  * RMSE          : 1.9821
  * MAE           : 1.4977
  * WAPE (%)      : 47.84%


In [3]:
# Step 2: Part B — Enterprise Probabilistic Demand Range Deliverable
hybrid_plan = pd.read_csv('n02ba_hybrid_supply_chain_plan.csv')

print("==========================================================================")
print(f"  PART B: ENTERPRISE PROBABILISTIC DEMAND RANGE DELIVERABLE ({TARGET_DRUG})")
print("==========================================================================")
print("First 10 Days Actionable Pack Order Ranges:")
display(hybrid_plan[['Date', 'Actual Sales', 'Lean Lower Bound (P10)', 'Expected Demand Anchor (P50)', 'Upper Target Stock (P90)', 'Order Range (Lean P10 Pack Target)', 'Order Range (Expected P50 Pack Target)', 'Order Range (Safety P90 Pack Target)']].head(10))

service_level = np.mean(test_series.values <= hybrid_plan['Upper Target Stock (P90)'].values) * 100
print(f"\nDeliverable Performance Metrics:")
print(f"  * Achieved P90 Inventory Service Level: {service_level:.2f}% (Target >= 95%)")
print(f"  * Average Daily Uncertainty Range    : {hybrid_plan['Uncertainty Band Width (P90 - P10)'].mean():.2f} units/day")


  PART B: ENTERPRISE PROBABILISTIC DEMAND RANGE DELIVERABLE (N02BA)
First 10 Days Actionable Pack Order Ranges:


,Date,Actual Sales,Lean Lower Bound (P10),Expected Demand Anchor (P50),Upper Target Stock (P90),Order Range (Lean P10 Pack Target),Order Range (Expected P50 Pack Target),Order Range (Safety P90 Pack Target)
0,2019-01-01,0.00,0.38,1.80,8.84,1,2,9
1,2019-01-02,4.15,1.36,2.24,8.76,2,3,9
2,2019-01-03,6.25,1.26,2.32,8.62,2,3,9
3,2019-01-04,2.60,1.03,2.20,9.24,2,3,10
4,2019-01-05,3.10,0.87,2.48,9.21,1,3,10
5,2019-01-06,2.25,0.63,2.38,9.87,1,3,10
6,2019-01-07,0.00,0.80,2.24,8.59,1,3,9
7,2019-01-08,0.00,1.09,2.57,9.64,2,3,10
8,2019-01-09,2.00,1.33,2.90,9.38,2,3,10
9,2019-01-10,5.60,1.14,2.73,8.47,2,3,9



Deliverable Performance Metrics:
  * Achieved P90 Inventory Service Level: 99.29% (Target >= 95%)
  * Average Daily Uncertainty Range    : 8.04 units/day
